# Measured line array: reconstruction and MUSIC

The measured input contains seven pressure–velocity sensors at 2900 and 3000 Hz, position 5. PI-ISS uses 500 reverse steps, seed 3 and 32 virtual positions. Reconstruction uses measurements with added 10 dB Gaussian noise, drawn before setting the sampling seed. Seven-sensor MUSIC uses the measurements before this added noise. MUSIC uses 2950 Hz. The interference region is x in [−0.6, 0.6] m and z in [1.6, 1.8] m.

Run cells in order. Sampling counts are reverse-diffusion steps; checkpoint numbers identify training state.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'models').is_dir() and (p / 'sampler').is_dir() and (p / 'trainer').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the MRSM repository.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
OUTPUT = ROOT / 'outputs' / 'line_array'
OUTPUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family': 'Arial', 'font.size': 10})

def format_current_figure():
    def tick(value, pos):
        return '0' if abs(value) < 1e-12 else f'{value:g}'
    for ax in plt.gcf().axes:
        if ax.get_xscale() == 'linear':
            ax.xaxis.set_major_formatter(FuncFormatter(tick))
        if ax.get_yscale() == 'linear':
            ax.yaxis.set_major_formatter(FuncFormatter(tick))

print('Repository:', ROOT)
print('Device:', device)
from scipy.signal import hilbert


In [ ]:
import os
from sampler.utils import *
from trainer.utils import restore_checkpoint
import torch
import torch.nn as nn
import json
from collections import namedtuple
from tqdm import tqdm
from models import get_model, get_ema
from sampler import VESDE
import numpy as np
from einops import rearrange
data_name = 'line_array'
version = 'v33'
opt_path = './results/' + data_name + '_' + version + '/config.json'
with open(opt_path, 'r', encoding='utf-8') as f:
    opt = json.load(f)
    opt['num_channels'] = opt['num_components'] + opt['num_conditions']
opt['results_path'] = str(Path(opt_path).parent)
opt['device'] = device
OPT_class = namedtuple('OPT_class', opt.keys())
opt = OPT_class(**opt)
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
net = get_model(opt)
net.to(device)
ema = get_ema(net.parameters(), decay=opt.ema_rate)
optimizer = torch.optim.Adam(net.parameters(), lr=opt.lr, betas=(0.9, 0.999), eps=1e-08, weight_decay=0)
state = dict(optimizer=optimizer, model=net, ema=ema, epoch=0, iteration=0, loss_train=[])
ckpt_path = opt.results_path + '/checkpoint_600.pth'
if not Path(ckpt_path).is_file():
    raise FileNotFoundError(ckpt_path)
state = restore_checkpoint(ckpt_path, state, device)
net = state['model']
ema = state['ema']
ema.copy_to(net.parameters())
print('Model loaded from epoch', state['epoch'])


In [ ]:
COORD_GROUND_TRUTH = {'pos1': {'s1': (0.5, 0.57), 's2': (-0.5, 0.67)}, 'pos2': {'s1': (0.94, 0.57), 's2': (-0.5, 0.67)}, 'pos3': {'s1': (0.94, 0.57), 's2': (-0.98, 0.67)}, 'pos4': {'s1': (0.94, 1.17), 's2': (-0.98, 0.67)}, 'pos5': {'s1': (0.94, 1.17), 's2': (-0.98, 1.27)}, 'pos6': {'s1': (0.94, 1.17), 's2': (-0.5, 1.27)}, 'pos7': {'s1': (0.94, 1.17), 's2': (0.0, 1.27)}, 'pos8': {'s1': (0.94, 0.57), 's2': (0.0, 1.27)}}
EXP_DATA_PATH = 'data/Aligned_2900_3000_pos5.npy'
data_raw_exp = np.load(EXP_DATA_PATH)
filename_base = os.path.basename(EXP_DATA_PATH).replace('.npy', '')
parts = filename_base.split('_')
if len(parts) >= 4 and parts[0] == 'Aligned':
    f1 = int(parts[1])
    f2 = int(parts[2])
    parsed_pos_str = parts[3]
    print(f'Frequencies: {f1} and {f2} Hz; position: {parsed_pos_str}')
else:
    print(f"Warning: unrecognized filename '{filename_base}'; using default frequencies and position.")
    (f1, f2) = (2800, 2900)
    parsed_pos_str = 'pos7'
f_target = (f1 + f2) / 2.0
c = 343.0
Z0 = 1.21 * c
k = 2 * np.pi * f_target / c
data_norm_exp = data_raw_exp.copy()
data_norm_exp[:, 1:, :] *= Z0
print(f'Velocity channels scaled by acoustic impedance: {Z0:.2f}.')
inv_scale_np = np.array([1.0, 1.0 / Z0, 1.0 / Z0, 1.0 / Z0]).reshape(1, 1, 4, 1)

def scalar_inv(x):
    if isinstance(x, torch.Tensor):
        dev = x.device
        dt = x.dtype
        inv_scale_t = torch.tensor(inv_scale_np, device=dev, dtype=dt)
        return x * inv_scale_t
    else:
        return x * inv_scale_np

def add_awgn_noise_per_channel(signal, snr_db):
    if snr_db is None:
        print('No added noise; using the measured signal.')
        return signal.copy()
    noisy_signal = np.zeros_like(signal)
    (num_sensors, num_channels, num_time) = signal.shape
    print(f'Adding Gaussian white noise at {snr_db} dB SNR.')
    for s in range(num_sensors):
        for c_idx in range(num_channels):
            clean_sig = signal[s, c_idx, :]
            sig_power = np.mean(clean_sig ** 2)
            if sig_power == 0:
                noisy_signal[s, c_idx, :] = clean_sig
                continue
            noise_power = sig_power / 10 ** (snr_db / 10)
            noise = np.random.normal(0, np.sqrt(noise_power), clean_sig.shape)
            noisy_signal[s, c_idx, :] = clean_sig + noise
    return noisy_signal
TARGET_SNR = 10
data_noisy_exp = add_awgn_noise_per_channel(data_norm_exp, TARGET_SNR)
m = 2
T = 10
T_prime_init = 1000
start_t = 2000
B_block = int(np.ceil((T_prime_init - m) / (T - m)))
T_prime = B_block * (T - m) + m
data_transposed = data_noisy_exp.transpose(2, 1, 0)
data_sliced = data_transposed[start_t:start_t + T_prime, :, :]
y_obs = data_sliced[np.newaxis, ...]
print(f'Observation shape: {y_obs.shape}')


## PI-ISS reconstruction


In [ ]:
outer_loop = 0
denoising_steps = 0
denoising_steps2 = 500
cof = 0.0001
seed_sample = 3
obs_type = 'regular'
scale = 5
alpha_case = 0.01
beta = 0.1
gamma = 0.5
physics_params = {'physics_c': 343.0, 'physics_rho0': 1.21, 'physics_dt': 1e-05, 'physics_dx': 0.032, 'reg_coef_wave': 10.0 * cof, 'reg_coef_cont': 0 * cof, 'reg_coef_euler': 0 * cof, 'reg_coef_bc': 0 * cof, 'normalized_velocity_zero': None}
sde2 = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=denoising_steps2)
net.eval()
reap_fn = lambda x, num: torch.stack([x for _ in range(num)])

def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
with torch.no_grad():
    if obs_type.lower() == 'regular':
        T_prime_y = T_prime
        transform_gen = transform = lambda x: x[:, :T_prime_y, 0:4, ::scale]
        sparsity = 1 - 1 / scale
    alpha = alpha_case / np.sqrt(1 - sparsity)
    y = y_obs[:, :, 0:4, :]
    print(f"Model output channels: {opt.num_components}.")
    print(f'Observation shape: {y.shape} -> (Batch, Time, four channels, seven sensors)')
    latent_mask = torch.ones([T, 1, 1]).float().to(device)
    obs_mask = torch.zeros([T, 1, 1]).float().to(device)
    frame_indices = torch.arange(T).to(device)
    net_fn = lambda x, t: net(x, x0=x, timesteps=t, latent_mask=reap_fn(latent_mask, len(x)), obs_mask=reap_fn(obs_mask, len(x)), frame_indices=reap_fn(frame_indices, len(x)))[0]
    print('Sampling PI-ISS for pressure and three velocity channels...')
    setup_seed(seed_sample)
    x_inpainted2 = s3gm_sample_1d_ISS_1dtube_res(opt, net_fn, sde2, y, transform_gen, alpha=0.1 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime, T_prime=T_prime, overlap=m, denoising_steps=denoising_steps2, eta=0, reg_coef_pa=1.0, data_scalar=scalar_inv, physics_start_step=denoising_steps2, **physics_params, corrector=NoneCorrector, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)
x_pred2 = x_inpainted2
x_pred2_phys = scalar_inv(x_inpainted2)[0]


## Full-vector MUSIC


In [ ]:
raw_7_phys = data_raw_exp[:, :, start_t:start_t + T_prime]
recon_32_phys = x_pred2_phys.transpose(2, 1, 0)
sensor_coords_7 = np.zeros((7, 3))
sensor_coords_7[:, 0] = 0.48 - np.arange(7) * 0.16
sensor_coords_32 = np.zeros((32, 3))
sensor_coords_32[:, 0] = 0.48 - np.arange(32) * 0.032

def run_music_pv(data_phys, coords, f_target, apply_z0=True):
    c = 343.0
    k = 2 * np.pi * f_target / c
    num_sens = coords.shape[0]
    data_scaled = data_phys.copy()
    if apply_z0:
        Z0 = 1.21 * c
        data_scaled[:, 1:, :] *= Z0
    data_analytic = hilbert(data_scaled, axis=-1)
    X = data_analytic.reshape(num_sens * 4, -1)
    R = X @ X.conj().T / X.shape[1]
    (eigvals, eigvecs) = np.linalg.eigh(R)
    Un = eigvecs[:, np.argsort(eigvals)[::-1]][:, 2:]
    Un_UnH = Un @ Un.conj().T
    scan_points = 150
    scan_x = np.linspace(-1.5, 1.5, scan_points)
    scan_z = np.linspace(0.01, 1.8, scan_points)
    (X_grid, Z_grid) = np.meshgrid(scan_x, scan_z)
    P_music = np.zeros_like(X_grid, dtype=float)
    R_mat = np.eye(3)
    for i in range(scan_points):
        for j in range(scan_points):
            r_s = np.array([X_grid[i, j], 1.0, Z_grid[i, j]])
            a_vec = np.zeros(num_sens * 4, dtype=complex)
            for m_s in range(num_sens):
                diff = coords[m_s] - r_s
                dist = np.linalg.norm(diff)
                u_local = R_mat @ (diff / dist)
                prop = np.exp(-1j * k * dist) / dist
                a_vec[m_s * 4:(m_s + 1) * 4] = prop * np.array([1.0, u_local[0], u_local[1], u_local[2]])
            a_vec /= np.linalg.norm(a_vec)
            P_music[i, j] = 1.0 / (np.abs(a_vec.conj().T @ Un_UnH @ a_vec) + 1e-12)
    return (X_grid, Z_grid, P_music)
(X_grid, Z_grid, P_7) = run_music_pv(raw_7_phys, sensor_coords_7, f_target)
(_, _, P_32) = run_music_pv(recon_32_phys, sensor_coords_32, f_target)
P_7_dB = 10 * np.log10(P_7 / P_7.max())
P_32_dB = 10 * np.log10(P_32 / P_32.max())
(fig, axes) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for (ax, values, title) in zip(axes, [P_7_dB, P_32_dB], ['Seven physical sensors', '32 virtual positions']):
    im = ax.contourf(X_grid, Z_grid, values, levels=np.linspace(-20, 0, 101), cmap='jet', extend='min')
    for (name, position) in COORD_GROUND_TRUTH[parsed_pos_str].items():
        ax.scatter(*position, marker='x', c='black')
    ax.set(title=title, xlabel='x (m)', ylabel='z (m)', aspect='equal')
fig.colorbar(im, ax=axes, label='Normalized spectrum (dB)')
format_current_figure()
fig.savefig(OUTPUT / 'music_comparison.pdf', bbox_inches='tight')
format_current_figure()
plt.show()
np.savez_compressed(OUTPUT / 'music_maps.npz', x=X_grid, z=Z_grid, sparse=P_7_dB, dense=P_32_dB)


## Local interference region


In [ ]:
(x_min, x_max) = (-0.6, 0.6)
(z_min, z_max) = (1.6, 1.8)
mask_red_box = (X_grid >= x_min) & (X_grid <= x_max) & (Z_grid >= z_min) & (Z_grid <= z_max)
psll_7_box = np.max(P_7_dB[mask_red_box])
psll_32_box = np.max(P_32_dB[mask_red_box])
psll_drop_box = psll_7_box - psll_32_box
print('-' * 50)
print(f'Interference region: x=[{x_min}, {x_max}], z=[{z_min}, {z_max}]')
print(f'Seven-sensor peak interference: {psll_7_box:.2f} dB')
print(f'32-position virtual-array peak interference: {psll_32_box:.2f} dB')
print(f'Peak interference reduction: {psll_drop_box:.2f} dB')
print('-' * 50)
pd.DataFrame({'array': ['physical_7', 'virtual_32'], 'regional_peak_db': [psll_7_box, psll_32_box]}).to_csv(OUTPUT / 'regional_peak.csv', index=False)
